In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import src
import pandas as pd

/Users/gperaza/Research/informal-jobs-model/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


We load the databases generated in Notebook 01.

In [2]:
enoe = pd.read_parquet(ROOT / "outputs/enoe_workers.parquet")
od = pd.read_parquet(ROOT / "outputs/od_workers.parquet")

For standardization purposes, the attributes, their representation in each survey, and their standardized results are as follows:

* **Gender**:

  * **ENOE**: `sex`
    * (Hombre) $\to$ `H`
    * (Mujer) $\to$ `F`
  * **OD**: `sexo_nacimiento`
    * (Hombres) $\to$ `H`
    * (Mujeres) $\to$ `F`
  * **Standardized attribute:** A `genero` column is created with values `H`, `F`, or `no_especificado`.

* **Occupation**:

  * **ENOE**: `pos_ocu`

    * (Trabajadores subordinados y remunerados) $\to$ `trabajador`
    * (Empleadores) $\to$ `trabajador`
    * (Trabajadores por cuenta propia) $\to$ `independiente`
    * (Trabajadores sin pago) $\to$ `sin_pago` (ENOE-only level; the OD has no unpaid-worker answer)
    * (No especificado) $\to$ `no_especificado`
  * **OD**: `ocupacion_raw`

    * (Empleado) $\to$ `trabajador`
    * (Persona trabajadora del hogar) $\to$ `trabajador`
    * (Trabajador del campo) $\to$ `trabajador`
    * (Profesor) $\to$ `trabajador`
    * (Patrón o empresario) $\to$ `trabajador`
    * (Trabajador independiente) $\to$ `independiente`
    * (Hogar) $\to$ `no_especificado`
    * (Estudiante) $\to$ `no_especificado`
    * (Jubilado o pensionado) $\to$ `no_especificado`
    * (Desempleado) $\to$ `no_especificado`
  * Respondents with a non-working status (Hogar, Estudiante, Jubilado, Desempleado) who nevertheless report having worked last week are kept as workers — they do work, but their position in the occupation is unknown — so they receive `no_especificado`.
  * **Standardized attribute:** An `ocupacion` column is created with values `trabajador`, `independiente`, `otro`, or `no_especificado`.

* **Age**:

  * **ENOE**: `eda`
  * **OD**: `edad`
  * **Standardization for both datasets**:

    * (0-2 years) $\to$ `0_2`
    * (3-4 years) $\to$ `3_4`
    * (5 years) $\to$ `5`
    * (6-7 years) $\to$ `6_7`
    * (8-11 years) $\to$ `8_11`
    * (12-14 years) $\to$ `12_14`
    * (15-17 years) $\to$ `15_17`
    * (18-24 years) $\to$ `18_24`
    * (25-49 years) $\to$ `25_49`
    * (50-59 years) $\to$ `50_59`
    * (60-64 years) $\to$ `60_64`
    * (65 years or more) $\to$ `65_y_mas`
  * **Standardized attribute:** An `edad_num` column is created with the numerical age, and an `edad_cat` column is created with values `0_2`, `3_4`, `5`, `6_7`, `8_11`, `12_14`, `15_17`, `18_24`, `25_49`, `50_59`, `60_64`, or `65_y_mas`.

* **Education**:

  * **ENOE**: `cs_p13_1`

    * (Ninguna) $\to$ `sin_instruccion`
    * (Preescolar) $\to$ `sin_instruccion`
    * (Primaria) $\to$ `primaria_o_secundaria`
    * (Secundaria) $\to$ `primaria_o_secundaria`
    * (Preparatoria o bachillerato) $\to$ `carrera_tecnica_o_preparatoria`
    * (Normal) $\to$ `carrera_tecnica_o_preparatoria`
    * (Carrera técnica) $\to$ `carrera_tecnica_o_preparatoria`
    * (Profesional) $\to$ `licenciatura`
    * (Maestría) $\to$ `postgrado`
    * (Doctorado) $\to$ `postgrado`
    * (No sabe) $\to$ `no_especificado`
  * **OD**: `escolaridad_raw`

    * (Ninguno) $\to$ `sin_instruccion`
    * (Kinder) $\to$ `sin_instruccion`
    * (Preescolar) $\to$ `sin_instruccion`
    * (Primaria) $\to$ `primaria_o_secundaria`
    * (Secundaria) $\to$ `primaria_o_secundaria`
    * (Normal básica) $\to$ `carrera_tecnica_o_preparatoria`
    * (Preparatoria o bachillerato) $\to$ `carrera_tecnica_o_preparatoria`
    * (Carrera técnica con secundaria) $\to$ `carrera_tecnica_o_preparatoria`
    * (Carrera técnica con preparatoria) $\to$ `carrera_tecnica_o_preparatoria`
    * (Licenciatura o profesional) $\to$ `licenciatura`
    * (Maestría o doctorado) $\to$ `postgrado`
    * (No sabe) $\to$ `no_especificado`
  * **Standardized attribute:** An `escolaridad` column is created with values `sin_instruccion`, `primaria_o_secundaria`, `carrera_tecnica_o_preparatoria`, `licenciatura`, `postgrado`, or `no_especificado`.

* **Municipality**:

  * **ENOE**: `mun`

    * (Guadalajara) $\to$ `guadalajara`
    * (Zapopan) $\to$ `zapopan`
    * (San Pedro Tlaquepaque) $\to$ `tlaquepaque`
    * (Tlajomulco de Zúñiga) $\to$ `tlajomulco`
    * (Tonalá) $\to$ `tonala`
    * (El Salto) $\to$ `el_salto`
    * (Juanacatlán) $\to$ `juanacatlan`
    * (Ixtlahuacán de los Membrillos) $\to$ `ixtlahuacan_membrillos`
    * (Zapotlanejo) $\to$ `zapotlanejo`
    * (Other municipalities) $\to$ `otro`
  * **OD**: `municipio_raw`

    * (Guadalajara) $\to$ `guadalajara`
    * (Zapopan) $\to$ `zapopan`
    * (San Pedro Tlaquepaque) $\to$ `tlaquepaque`
    * (Tlajomulco de Zúñiga) $\to$ `tlajomulco`
    * (Tonalá) $\to$ `tonala`
    * (El Salto) $\to$ `el_salto`
    * (Juanacatlán) $\to$ `juanacatlan`
    * (Ixtlahuacán de los Membrillos) $\to$ `ixtlahuacan_membrillos`
    * (Zapotlanejo) $\to$ `zapotlanejo`
    * (Other municipalities) $\to$ `otro`
  * **Standardized attribute:** A `municipio` column is created with standardized municipality names for the municipalities of interest, while all remaining municipalities are classified as `otro`.

* **Marital status**:

  * **ENOE**: `e_con`

    * (Unión libre) $\to$ `union_libre`
    * (Separado) $\to$ `separado`
    * (Divorciado) $\to$ `divorciado`
    * (Viudo) $\to$ `viudo`
    * (Casado) $\to$ `casado`
    * (Soltero) $\to$ `soltero`
  * **OD**: `estado_civil_raw`

    * (Unión libre) $\to$ `union_libre`
    * (Separado) $\to$ `separado`
    * (Divorciado) $\to$ `divorciado`
    * (Viudo) $\to$ `viudo`
    * (Casado) $\to$ `casado`
    * (Soltero) $\to$ `soltero`
  * **Standardized attribute:** An `estado_civil` column is created with values `union_libre`, `separado`, `divorciado`, `viudo`, `casado`, `soltero`, or `no_especificado`.

* **Relationship to household head**:

  * **ENOE**: `par_c`

    * (Head of household) $\to$ `jefe_del_hogar`
    * (Spouse or partner) $\to$ `conyuge`
    * (Son or daughter) $\to$ `hijo`
    * (Other relative) $\to$ `otro_parentesco`
    * (Non-relative) $\to$ `sin_parentesco`
  * **OD**: `parentesco_raw`

    * (Jefe del hogar) $\to$ `jefe_del_hogar`
    * (Cónyuge) $\to$ `conyuge`
    * (Compañero) $\to$ `conyuge`
    * (Hijo) $\to$ `hijo`
    * (Nieto) $\to$ `otro_parentesco`
    * (Otro parentesco) $\to$ `otro_parentesco`
    * (Sin parentesco) $\to$ `sin_parentesco`
  * **Standardized attribute:** A `parentesco` column is created with values `jefe_del_hogar`, `conyuge`, `hijo`, `otro_parentesco`, `sin_parentesco`, or `no_especificado`.

* **Dwelling size**:

  * **ENOE**: `dwelling_size`, the number of habitual/new residents (`c_res` 1 or 3) in the dwelling counted over the full SDEM roster (all ages, all households).
  * **OD**: `dwelling_size`, the self-reported number of persons living permanently in the dwelling (`personas_en_vivienda`, "1" … "10 y +").
  * **Standardized attributes:** `tamano_viv_num` (capped at 10 in both surveys, since the OD answer stops there) and `tamano_viv_cat` with levels `1` … `6`, `7_y_mas` or `no_especificado`. The categories are collapsed at 7+ because the ENOE roster count and the OD self-report diverge strongly in the tail (8+ persons: 8.4% vs 1.6% of workers before the change).

* **Economic sector**:

  The code → class mapping for both surveys lives in `src/mappings/sector.yaml` (loaded by `src.load_mapping("sector")`), where each SCIAN code is annotated with its INEGI label and the changelog records reassignments. Summary:

  * **ENOE**: `scian` (2-digit SCIAN sector)
    * Agricultura (1), Gobierno (20) $\to$ `gobierno_otro_agricultura`
    * Minería (2), Electricidad/agua/gas (3), Construcción (4), Manufactura (5) $\to$ `manufactura_construccion`
    * Comercio mayoreo/menudeo (6, 7) $\to$ `comercio`
    * Transportes (8), Información en medios (9), Servicios (10–19) $\to$ `servicios_transporte`
    * No especificado (21) $\to$ `no_especificado`
  * **OD**: `giro_empresa`
    * (Comercio) $\to$ `comercio`
    * (Servicio), (Educación) $\to$ `servicios_transporte`
    * (Industria) $\to$ `manufactura_construccion`
    * (Gobierno/sector público) $\to$ `gobierno_otro_agricultura`
  * **Standardized attribute:** A `sector` column with the four classes or `no_especificado`, plus a boolean `sector_desconocido`.

* **Place of work** (`lugar_trabajo`, added in Phase 4):

  * **ENOE**: COE1 section IV (`p4`, `p4b`, `p4e`, `p4f`, `p4h`): premises of a non-commercial unit or an institution $\to$ `establecimiento`; premises in the commerce sector or a fixed / semi-fixed / improvised stall $\to$ `comercio_o_puesto`; employer's or client's home and domestic workers $\to$ `otra_vivienda`; field, itinerant, vehicle, own home, construction site, visiting clients $\to$ `otro_o_sin_local`.
  * **OD**: most frequent destination type of the work trips (`destino_trabajo`): factory/workshop, office, hospital, school, restaurant, cultural or sports venue $\to$ `establecimiento`; "Comercio, mercado, tienda o centro comercial" $\to$ `comercio_o_puesto`; "Otra vivienda" $\to$ `otra_vivienda`; "Su casa", "Otros" $\to$ `otro_o_sin_local`; no work trip on the survey day $\to$ `no_especificado` (marginalized at scoring).
  * The OD answer for commerce covers both shops and market stalls, which is why ENOE stalls are grouped with commercial premises rather than with "no premises".

* **Household roster** (`hogar_trabajadores_cat`, `hogar_ninos_cat`): employed members and children aged 6–11 in the household, from both rosters (the OD roster starts at age 6). Comparable across surveys but without predictive value (Phase 4.3), so kept as data, not as model features.

* **Informality**:

  * **ENOE**: `emp_ppal`

    * (Empleo informal) $\to$ `1`
    * (Empleo formal) $\to$ `0`
  * **OD**: Not available
  * **Standardized attribute:** An `informal` column is created only for ENOE, where `1` indicates an informal worker and `0` indicates a formal worker.


**Fallback paths.** Every mapping fails loudly on an unknown source category (`src.assert_mapping_covers`); the only values allowed to fall through are: ENOE `e_con == 9` ("no sabe") and OD `estado_civil_raw == "Otros (especifique)"` → `no_especificado`; ENOE `eda == 98` (age unspecified) → missing age; source item non-response (NaN) in any OD attribute → `no_especificado`; OD `escolaridad_raw == "No sabe"` is mapped to `no_especificado` together with non-response (a possible refinement is to keep it as its own level). Municipalities outside the metropolitan area are `otro`; a missing municipality code is `no_especificado`.


In [3]:
enoe_harmonized = src.harmonize_enoe_dataframe(enoe)
od_harmonized = src.harmonize_od_dataframe(od)

In [4]:
common_columns = ["genero", "ocupacion", "edad_num", "edad_cat", "escolaridad", "municipio", "estado_civil", "parentesco", "tamano_viv_num", "tamano_viv_cat", "sector", "sector_desconocido"]

assert enoe_harmonized.shape[0] == enoe.shape[0], "ENOE row count changed during harmonization."
assert od_harmonized.shape[0] == od.shape[0], "OD row count changed during harmonization."
assert set(common_columns).issubset(enoe_harmonized.columns), "Missing harmonized columns in ENOE."
assert set(common_columns).issubset(od_harmonized.columns), "Missing harmonized columns in OD."
assert "informal" in enoe_harmonized.columns, "Missing informal label in ENOE."

print("Harmonization validation completed successfully.")

Harmonization validation completed successfully.


In [5]:
for column in common_columns:
    print(f"\n{column}")
    print("ENOE:", enoe_harmonized[column].value_counts(dropna=False).to_dict())
    print("OD:", od_harmonized[column].value_counts(dropna=False).to_dict())


genero
ENOE: {'H': 30306, 'F': 21401}
OD: {'H': 16684, 'F': 10229}

ocupacion
ENOE: {'trabajador': 41123, 'independiente': 9332, 'sin_pago': 1252}
OD: {'trabajador': 23274, 'independiente': 3417, 'no_especificado': 222}

edad_num
ENOE: {np.int64(26): 1351, np.int64(30): 1346, np.int64(27): 1343, np.int64(28): 1342, np.int64(29): 1331, np.int64(23): 1303, np.int64(24): 1289, np.int64(50): 1265, np.int64(32): 1262, np.int64(25): 1221, np.int64(36): 1220, np.int64(33): 1205, np.int64(38): 1202, np.int64(22): 1199, np.int64(31): 1188, np.int64(40): 1174, np.int64(37): 1167, np.int64(35): 1150, np.int64(34): 1137, np.int64(39): 1136, np.int64(41): 1122, np.int64(21): 1113, np.int64(20): 1105, np.int64(51): 1103, np.int64(43): 1098, np.int64(46): 1056, np.int64(42): 1036, np.int64(45): 1017, np.int64(52): 1002, np.int64(49): 995, np.int64(44): 973, np.int64(53): 933, np.int64(48): 932, np.int64(47): 909, np.int64(19): 903, np.int64(55): 860, np.int64(54): 849, np.int64(18): 834, np.int64(56

Since we are going to some classifications in the pipeline, it is important to analyze the distribution of those attributes with unspecified values or marked as “other.”

In [6]:
print("ENOE municipalities mapped as 'otro':")
print(enoe_harmonized.loc[enoe_harmonized["municipio"] == "otro", "mun"].value_counts(dropna=False))

print("\nOD municipalities mapped as 'otro':")
print(od_harmonized.loc[od_harmonized["municipio"] == "otro", "municipio_raw"].value_counts(dropna=False))

print("OD education mapped as 'no_especificado':")
print(od_harmonized.loc[od_harmonized["escolaridad"] == "no_especificado", "escolaridad_raw"].value_counts(dropna=False))

print("OD sectors mapped as 'no_especificado':")
print(od_harmonized.loc[od_harmonized["sector_desconocido"], "giro_empresa"].value_counts(dropna=False))

print("\nOD workers with a non-working status (ocupacion mapped to 'no_especificado'):")
print(od_harmonized.loc[od_harmonized["ocupacion"] == "no_especificado", "ocupacion_raw"].value_counts(dropna=False))
print("\nENOE occupation levels:", enoe_harmonized["ocupacion"].value_counts().to_dict())
print("OD occupation levels:", od_harmonized["ocupacion"].value_counts().to_dict())

ENOE municipalities mapped as 'otro':
mun
67     984
93     946
23     796
18     750
82     584
53     577
123    529
13     441
37     372
73     356
6      340
100    330
47     313
66     304
22     302
36     300
50     292
109    270
45     261
46     257
94     240
83     194
35     182
10     175
27     167
80     165
81     157
63     143
1      141
2      139
108    138
15     130
74     127
68     119
8      111
119    105
4      104
85      94
16      93
30      90
59      78
43      73
58      67
110     58
116     56
77      54
118     50
57      41
5       40
96      38
121     33
84      26
105     25
102     17
Name: count, dtype: Int64

OD municipalities mapped as 'otro':
Series([], Name: count, dtype: int64[pyarrow])
OD education mapped as 'no_especificado':
escolaridad_raw
<NA>       5037
No sabe     158
Name: count, dtype: int64[pyarrow]
OD sectors mapped as 'no_especificado':
giro_empresa
<NA>    9484
Name: count, dtype: int64[pyarrow]

OD workers with a non-worki

In this distribution, we can see that, for ENOE, there is a considerable number of observations whose municipalities are classified as `other`, since they are not part of the set of municipalities approved by the OD. For this reason, the comparative and classification analysis between the two databases will be conducted by considering only the geography common to both sources.

Furthermore, in the education variable, there are 5,195 observations from the OD classified as `unspecified`. This result will be relevant during the construction of the machine learning models, as it will allow us to evaluate the contribution of educational attainment to model performance and to define how to handle observations for which this information is not available.

Finally, the 9,484 observations from the OD whose economic sector was classified as `no_specified` correspond to workers whose sector must be imputed later using the machine learning model, assigning them to one of the previously validated economic sectors.

In [7]:
print(f"ENOE workers: {len(enoe_harmonized):,}")
print(f"OD workers: {len(od_harmonized):,}")
print(f"ENOE unknown sectors: {enoe_harmonized['sector_desconocido'].sum():,}")
print(f"OD unknown sectors: {od_harmonized['sector_desconocido'].sum():,}")

ENOE workers: 51,707
OD workers: 26,913
ENOE unknown sectors: 218
OD unknown sectors: 9,484


In [8]:
output_directory = ROOT / "outputs"
output_directory.mkdir(exist_ok=True)

enoe_harmonized.to_parquet(output_directory / "enoe_harmonized.parquet", index=False)
od_harmonized.to_parquet(output_directory / "od_harmonized.parquet", index=False)